### 📖 How to Use the ML Score Converter

Welcome! This tool automatically converts your raw questionnaire scores into Machine Learning (ML) transformed scores. It is fully automated and requires no coding experience. Just follow these simple steps!

#### **Part 1: Prepare Your Data**
Before running the converter, please ensure your dataset is saved as an Excel (`.xlsx`) or CSV (`.csv`) file and formatted correctly.

* **ID Column:** Your file must include an `id` column to identify each record.
* **Item Columns:** The tool supports either **10-item** or **16-item** versions.
* **Column Naming Rule:** The item column headers *must* start with their corresponding specific "D-code" format (e.g., `D1.1`, `D2.2`). The program relies on these codes to accurately match your data with the model's features. (For example: `D1.1 Concentration`, `D1.6 Conversation`).

💡 **Need a reference?** Please click the links below to view or download the exact data templates:
* 🔗 [Data example_10-item.xlsx](https://docs.google.com/spreadsheets/d/11Eu28wQQjHdtQrCw6lsEuCG4YQaopeHD/edit?usp=sharing&ouid=104468075387492268190&rtpof=true&sd=true)
* 🔗 [Data example_16-item.xlsx](https://docs.google.com/spreadsheets/d/17KAsZrdP6h0F1o_Dp6iDTSlw6aX5YjVz/edit?usp=sharing&ouid=104468075387492268190&rtpof=true&sd=true)

---

#### **Part 2: Run the Converter**
Once your data is ready, follow these steps to get your transformed scores:

1. **Run the Code:** Open the script (or Google Colab notebook) and click the **"Play"** button on the left side of the code block to execute it.
2. **Upload Your File:** When the code runs, a **"Choose Files"** button will appear. Click it and select your prepared data file from your computer.
3. **Automatic Processing:** Sit back and wait a few seconds! The script is smart enough to:
   * Automatically detect whether your file contains 10 or 16 items.
   * Download the corresponding XGBoost ML model.
   * Map your columns and calculate the scores.
4. **Download Results:** Upon completion, the tool will automatically download a new Excel file to your computer (e.g., `transformed_output_10items.xlsx`). Open this file, and you will find a brand new column at the very end called **`final_score`**!

In [3]:
# 1. Install and import necessary packages
!pip install xgboost pandas openpyxl gdown -q

import pandas as pd
import xgboost as xgb
import gdown
import re
from google.colab import files

def main():
    # ==========================================
    # Configuration Area: Google Drive links and model features
    # ==========================================
    MODEL_CONFIG = {
        10: {
            'url': 'https://drive.google.com/file/d/17Yqc2V9sz8LK0bEgWJrABS5jtuqey9U7/view?usp=sharing',
            'features': ["D11", "D22", "D32", "D41", "D52", "D61", "D53", "D23", "D16", "D21"]
        },
        16: {
            'url': 'https://drive.google.com/file/d/1_6oGc1s3O6Csmo6nr6W-nM68KfII1XRT/view?usp=sharing',
            'features': ["D11", "D22", "D32", "D41", "D52", "D61", "D53", "D23", "D16", "D21", "D43", "D25", "D42", "D24", "D54", "D51"]
        }
    }

    # ==========================================
    # Step 1: Read the user-uploaded Excel or CSV file
    # ==========================================
    print("📂 Please upload the data file (.xlsx or .csv) containing your records:")
    uploaded = files.upload()

    if not uploaded:
        print("❌ No file uploaded. Program terminated.")
        return

    input_filename = list(uploaded.keys())[0]

    # Support reading both .csv and .xlsx files
    if input_filename.endswith('.csv'):
        df = pd.read_csv(input_filename)
    else:
        df = pd.read_excel(input_filename)

    print(f"✅ Successfully read file: {input_filename}")

    # ==========================================
    # Step 2: Automatically detect the number of items (detecting columns like D1.1, D2.2)
    # ==========================================
    # Find all columns starting with "D" followed by a digit, a dot, and a digit (e.g., D1.1)
    item_cols = [c for c in df.columns if re.match(r'^D\d\.\d', str(c))]
    num_items = len(item_cols)
    print(f"📊 Detected {num_items} item columns.")

    if num_items not in MODEL_CONFIG:
        print(f"❌ Error: Currently only supports 10-item or 16-item conversions (Detected {num_items} items in the file).")
        return

    config = MODEL_CONFIG[num_items]
    feature_names = config['features']

    # ==========================================
    # Step 3: Download the corresponding model
    # ==========================================
    print(f"📥 Downloading the {num_items}-item model file from Google Drive...")
    output_model_name = f'model_{num_items}.json'
    gdown.download(config['url'], output_model_name, fuzzy=True)

    print("⚙️ Loading the XGBoost model...")
    model = xgb.XGBRegressor()
    model.load_model(output_model_name)

    # ==========================================
    # Step 4: Data conversion and mapping (Handling feature name formatting)
    # ==========================================
    print("🔄 Extracting column codes and mapping to model features...")
    X_predict = pd.DataFrame()

    # Iterate through the item columns found in the dataset
    for col in item_cols:
        # Use regular expressions to extract "D1" and "1" from "D1.1 Concentration"
        match = re.match(r'^(D\d)\.(\d)', str(col))
        if match:
            # Combine them into the format required by the model, e.g., "D11"
            model_feat_name = match.group(1) + match.group(2)
            X_predict[model_feat_name] = df[col]

    # Check if any required model features are missing from the uploaded file
    for feat in feature_names:
        if feat not in X_predict.columns:
            print(f"⚠️ Warning: Missing column for model feature '{feat}'. Automatically filling with 0.")
            X_predict[feat] = 0

    # ⭐️ Crucial Step: Ensure the feature order matches EXACTLY with the model's training order!
    X_predict = X_predict[feature_names]

    # ==========================================
    # Step 5: Calculate scores and export the results
    # ==========================================
    print("🪄 Calculating ML transformed scores...")
    df['final_score'] = model.predict(X_predict)

    # Export the final dataframe to an Excel file
    output_filename = f'transformed_output_{num_items}items.xlsx'
    df.to_excel(output_filename, index=False)

    print(f"🎉 Conversion complete! You used the {num_items}-item version.")
    print(f"   Results have been saved to '{output_filename}' and will download automatically...")
    files.download(output_filename)

if __name__ == "__main__":
    main()

📂 Please upload the data file (.xlsx or .csv) containing your records:


Saving Data example_16-item.xlsx to Data example_16-item.xlsx
✅ Successfully read file: Data example_16-item.xlsx
📊 Detected 16 item columns.
📥 Downloading the 16-item model file from Google Drive...


Downloading...
From: https://drive.google.com/uc?id=1_6oGc1s3O6Csmo6nr6W-nM68KfII1XRT
To: /content/model_16.json
100%|██████████| 542k/542k [00:00<00:00, 89.1MB/s]

⚙️ Loading the XGBoost model...
🔄 Extracting column codes and mapping to model features...
🪄 Calculating ML transformed scores...
🎉 Conversion complete! You used the 16-item version.
   Results have been saved to 'transformed_output_16items.xlsx' and will download automatically...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>